# Block 5 — Dialogue Network Analysis

**Input:** `utterance_features.csv`  
**Outputs:** `network_node_summary.csv` · `network_edges.csv` · `figures/dialogue_network.png` · `figures/centrality_bar.png`  
**Method:** Directed weighted graph where each edge (A → B) represents A addressing B.  
**Node metrics:** betweenness centrality · degree centrality · in-degree · out-degree · clustering coefficient  
**Edge attributes:** utterance_count (weight) · mean_sentiment · dominant_emotion

## 0. Params — change only this cell

In [1]:
from pathlib import Path

FILM_ID        = "toy_story_1995"
PROTAGONIST    = "Woody"   # canonical_name of the film's protagonist (Buzz is co-lead)
REMOVAL_TARGET = None      # §12 keystone to remove; None = auto-pick highest-betweenness non-lead
YEAR           = 1995
DATA_ROOT      = Path("../data/02_processed")
FEAT_ROOT      = Path("../data/04_features")
MIN_EDGE_COUNT = 3

ALIAS = {
}

# Gender per character — pulled automatically from characters.csv for this FILM_ID,
# keyed by the SAME canonical-name normalization the network uses (strip + title),
# with ALIAS applied. Only M / F kept; anything else is omitted (homophily skips it).
import pandas as _pd
CHARS_FILE = DATA_ROOT / FILM_ID / f"character_review_{FILM_ID}.csv"  # per-film reviewed file
_fc = _pd.read_csv(CHARS_FILE, dtype=str).fillna("")

# Node identity is the UNIQUE character_id (names can collide / title-case can merge
# distinct characters), so GENDER is keyed by character_id, not by name.
GENDER = {}
for _idcol in ("review_character_id", "parser_character_id"):
    if _idcol in _fc.columns:
        GENDER.update({cid: g for cid, g in zip(_fc[_idcol], _fc["gender"])
                       if cid and g in ("M", "F")})

# ── Network inclusion + id-level merges, both taken from the reviewed file ────
# included_in_network gates which characters become nodes; merge_into_character_id
# folds alias rows (e.g. "Pabbie" -> "Grand Pabbie") into their canonical id.
_inc_id = "review_character_id" if "review_character_id" in _fc.columns else "parser_character_id"
INCLUDED_IDS = {
    cid for cid, inc in zip(_fc[_inc_id], _fc.get("included_in_network", ""))
    if cid and str(inc).strip().lower() in ("true", "1", "yes")
}
MERGE_MAP = {}
if "merge_into_character_id" in _fc.columns:
    MERGE_MAP = {
        cid: tgt.strip()
        for cid, tgt in zip(_fc[_inc_id], _fc["merge_into_character_id"].astype(str))
        if cid and tgt.strip()
    }
# merges compose with any manual ALIAS above (manual ALIAS wins on conflict)
ALIAS = {**MERGE_MAP, **ALIAS}

# canonical_name kept only for display; resolve the protagonist NAME -> its character_id
_name_to_id = {n: cid for n, cid in
               zip(_fc["canonical_name"].str.strip().str.title(), _fc["review_character_id"]) if cid}
PROTAGONIST_LABEL = PROTAGONIST
PROTAGONIST = _name_to_id.get(PROTAGONIST, PROTAGONIST)   # now a character_id

FILM_DIR    = DATA_ROOT / FILM_ID
# input: prefer the scene-level addressee file (notebook 07 output, the canonical one),
# fall back to the plain file only if _scene is absent
_scene = FILM_DIR / "utterances_with_addressee_scene.csv"
_plain = FILM_DIR / "utterances_with_addressee.csv"
INPUT_PATH  = _scene if _scene.exists() else _plain
FIGURES_DIR = FILM_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

PLOT_PATH                = FIGURES_DIR / "dialogue_network.png"
CENT_PATH                = FIGURES_DIR / "centrality_bar.png"
NODES_PATH               = FILM_DIR / "network_node_summary.csv"
EDGES_PATH               = FILM_DIR / "network_edges.csv"
PROTAGONIST_METRICS_PATH = FILM_DIR / "protagonist_metrics.csv"
HOMOPHILY_NULL_PATH      = FILM_DIR / "homophily_null.csv"
HOMOPHILY_NULL_FIG       = FIGURES_DIR / "homophily_null.png"
PROT_SAMESEX_FIG         = FIGURES_DIR / "protagonist_samesex_null.png"
BETWEENNESS_NULL_PATH    = FILM_DIR / "betweenness_null.csv"
BETWEENNESS_NULL_FIG     = FIGURES_DIR / "betweenness_null.png"
FILM_FEATURES_PATH       = FILM_DIR / "film_features.csv"

N_NULL_SAMPLES = 2000
NULL_SEED      = 42

def film_display_title(film_id):
    special = {
        "beautyandthebeast_1991": "Beauty and the Beast (1991)",
        "cars2": "Cars 2",
        "findingnemo": "Finding Nemo",
        "up": "Up",
    }
    if film_id in special:
        return special[film_id]
    stem, sep, year = film_id.rpartition("_")
    if sep and year.isdigit() and len(year) == 4:
        title = stem.replace("_", " ").title()
        for word in ("And", "The", "Of"):
            title = title.replace(f" {word} ", f" {word.lower()} ")
        return f"{title} ({year})"
    return film_id.replace("_", " ").title()

FILM_TITLE = film_display_title(FILM_ID)

print(f"Film            : {FILM_ID}")
print(f"Display title   : {FILM_TITLE}")
print(f"Protagonist     : {PROTAGONIST_LABEL}  (id: {PROTAGONIST})")
print(f"Input           : {INPUT_PATH}")
print(f"Min edge count  : {MIN_EDGE_COUNT}")
print(f"Figures dir     : {FIGURES_DIR}")

Film            : toy_story_1995
Display title   : Toy Story (1995)
Protagonist     : Woody  (id: toy_story_1995_woody)
Input           : ..\data\processed\toy_story_1995\utterances_with_addressee_scene.csv
Min edge count  : 3
Figures dir     : ..\data\processed\toy_story_1995\figures


## 1. Load data

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(INPUT_PATH)
print(f"Rows   : {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\naddressee_type distribution:")
print(df["addressee_type"].value_counts())

Rows   : 804
Columns: ['film_id', 'event_id', 'scene_id', 'utterance_id', 'scene_position_pct', 'speaker', 'raw_speaker', 'character_id', 'canonical_name', 'text', 'word_count', 'utterance_type', 'addressee', 'addressee_type', 'line_number', 'parenthetical', 'cue_type', 'gender', 'gender_rationale', 'is_named', 'is_protagonist', 'included_in_network', 'parser_canonical_name', 'confidence']

addressee_type distribution:
addressee_type
individual    455
unclear       159
monologue     100
group          72
non_human      18
Name: count, dtype: int64


## 2. Filter and clean

Keep only `addressee_type == "individual"` rows.  
Map `addressee` (character_id) → `canonical_name` using the speaker table already in the data.

In [3]:
# Build character_id → canonical_name from the dataset itself
id_to_name = (
    df[["character_id", "canonical_name"]]
    .dropna()
    .drop_duplicates("character_id")
    .set_index("character_id")["canonical_name"]
    .str.strip().str.title()
    .to_dict()
)

print(f"Unique character_ids mapped: {len(id_to_name)}")
for k, v in sorted(id_to_name.items())[:8]:
    print(f"  {k:45s} → {v}")

Unique character_ids mapped: 35
  toy_story_1995_alien                          → Alien
  toy_story_1995_aliens                         → Aliens
  toy_story_1995_andy                           → Andy
  toy_story_1995_attendant                      → Attendant
  toy_story_1995_bo_peep                        → Bo Peep
  toy_story_1995_buzz                           → Buzz
  toy_story_1995_female_voice_over_speaker      → Female Voice Over Speaker
  toy_story_1995_hamm                           → Hamm


In [4]:
dfi = df[df["addressee_type"] == "individual"].copy()
dfi["speaker_clean"]   = dfi["character_id"]   # unique character_id, not name
dfi["addressee_clean"] = dfi["addressee"]      # addressee column is already a character_id

# Optional id-level merge (ALIAS maps character_id -> character_id; empty for most films)
dfi["speaker_clean"]   = dfi["speaker_clean"].replace(ALIAS)
dfi["addressee_clean"] = dfi["addressee_clean"].replace(ALIAS)

dfi = dfi.dropna(subset=["speaker_clean", "addressee_clean"])
dfi = dfi[dfi["speaker_clean"] != dfi["addressee_clean"]]  # drop self-loops

# Keep only characters curated as network-included (review file). Films whose
# review predates this column (empty INCLUDED_IDS) keep the old behaviour.
if INCLUDED_IDS:
    dfi = dfi[
        dfi["speaker_clean"].isin(INCLUDED_IDS)
        & dfi["addressee_clean"].isin(INCLUDED_IDS)
    ]

print(f"Rows after filter + self-loop removal: {len(dfi)}")
print("\nSample (speaker -> addressee):")
for _, r in dfi[["speaker_clean","addressee_clean"]].head(8).iterrows():
    print(f"  {id_to_name.get(r.speaker_clean, r.speaker_clean):20s} -> {id_to_name.get(r.addressee_clean, r.addressee_clean)}")

Rows after filter + self-loop removal: 422

Sample (speaker -> addressee):
  Woody                -> Andy
  Andy                 -> Woody
  Woody                -> Andy
  Andy                 -> Woody
  Mrs. Davis           -> Andy
  Andy                 -> Mrs. Davis
  Andy                 -> Mrs. Davis
  Mrs. Davis           -> Andy


## 3. Build edge table

Aggregate by `(speaker_clean, addressee_clean)` and compute per-edge stats.  
Drop edges below `MIN_EDGE_COUNT`.

In [5]:
agg = (
    dfi.groupby(["speaker_clean", "addressee_clean"])
       .agg(utterance_count = ("utterance_id", "count"))
       .reset_index()
)

agg = agg[agg["utterance_count"] >= MIN_EDGE_COUNT].sort_values("utterance_count", ascending=False)
print(f"Edges retained (>= {MIN_EDGE_COUNT} utterances): {len(agg)}")
print()
print(agg.to_string(index=False))

Edges retained (>= 3 utterances): 24

                speaker_clean               addressee_clean  utterance_count
         toy_story_1995_woody           toy_story_1995_buzz              114
          toy_story_1995_buzz          toy_story_1995_woody               91
          toy_story_1995_andy      toy_story_1995_mrs_davis               22
     toy_story_1995_mrs_davis           toy_story_1995_andy               21
        toy_story_1995_slinky          toy_story_1995_woody               15
        toy_story_1995_hannah            toy_story_1995_sid               12
           toy_story_1995_rex          toy_story_1995_woody               11
       toy_story_1995_bo_peep          toy_story_1995_woody               11
         toy_story_1995_woody            toy_story_1995_sid               11
         toy_story_1995_woody         toy_story_1995_slinky               10
           toy_story_1995_sid         toy_story_1995_hannah               10
toy_story_1995_mr_potato_head         

## 4. Build directed graph

In [6]:
import networkx as nx

G = nx.DiGraph()
for _, r in agg.iterrows():
    G.add_edge(
        r["speaker_clean"], r["addressee_clean"],
        weight = r["utterance_count"],
        distance = 1.0 / r["utterance_count"],   # heavier ties = shorter path (unified with keystone/null)
    )

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print(f"Density            : {nx.density(G):.4f}")
print(f"Reciprocity        : {nx.reciprocity(G):.4f}  (fraction of edges with a return edge)")
print(f"Weakly connected   : {nx.is_weakly_connected(G)}")
print(f"Strongly connected : {nx.is_strongly_connected(G)}")

Nodes : 12
Edges : 24
Density            : 0.1818
Reciprocity        : 0.9167  (fraction of edges with a return edge)
Weakly connected   : True
Strongly connected : True


## 5. Node metrics

| Metric | Description |
|---|---|
| betweenness | Fraction of shortest paths passing through this node (weighted) |
| degree_centrality | Combined in+out degree / (n−1) |
| in_degree | How many characters speak *to* this character |
| out_degree | How many characters this character speaks *to* |
| clustering | Local clustering coefficient (on undirected projection) |

In [7]:
betweenness = nx.betweenness_centrality(G, normalized=True, weight="distance")  # distance=1/utterances, unified
degree_cent = nx.degree_centrality(G)
in_deg      = dict(G.in_degree())
out_deg     = dict(G.out_degree())
clustering  = nx.clustering(G.to_undirected(), weight="weight")

nodes   = sorted(G.nodes())
summary = pd.DataFrame({
    "character":         nodes,
    "name":              [id_to_name.get(n, n) for n in nodes],
    "betweenness":       [round(betweenness[n], 4) for n in nodes],
    "degree_centrality": [round(degree_cent[n],  4) for n in nodes],
    "in_degree":         [in_deg[n]  for n in nodes],
    "out_degree":        [out_deg[n] for n in nodes],
    "clustering":        [round(clustering[n], 4) for n in nodes],
}).sort_values("betweenness", ascending=False).reset_index(drop=True)

print("NODE SUMMARY — sorted by betweenness centrality")
print("=" * 72)
print(summary.to_string(index=False))

NODE SUMMARY — sorted by betweenness centrality
                    character            name  betweenness  degree_centrality  in_degree  out_degree  clustering
         toy_story_1995_woody           Woody       0.9455             1.5455          9           8      0.0079
          toy_story_1995_andy            Andy       0.1818             0.3636          2           2      0.0000
toy_story_1995_mr_potato_head Mr. Potato Head       0.1818             0.3636          2           2      0.0631
           toy_story_1995_sid             Sid       0.1818             0.3636          2           2      0.0000
          toy_story_1995_hamm            Hamm       0.0000             0.2727          1           2      0.0631
          toy_story_1995_buzz            Buzz       0.0000             0.2727          2           1      0.2199
       toy_story_1995_bo_peep         Bo Peep       0.0000             0.1818          1           1      0.0000
        toy_story_1995_hannah          Hannah   

## 6. Sanity checks

In [8]:
print("CHECK 1 — Edge count and weight distribution")
weights = [G[u][v]["weight"] for u,v in G.edges()]
print(f"  Total directed edges : {len(weights)}")
print(f"  Total utterances     : {sum(weights)}")
print(f"  Weight — min={min(weights)}  median={np.median(weights):.0f}  max={max(weights)}")

print("\nCHECK 2 — Reciprocity")
recip = nx.reciprocity(G)
print(f"  Reciprocity = {recip:.3f}")
if recip > 0.5:
    print("  NOTE: > 50% of edges have a return edge — dialogue is fairly mutual")
else:
    print("  NOTE: < 50% of edges are reciprocated — many one-sided speech flows")

print("\nCHECK 3 — Hub dominance")
top = summary.iloc[0]
print(f"  Top node by betweenness: {top.character}  ({top.betweenness:.3f})")
if top.betweenness > 0.5:
    print("  NOTE: single dominant hub (protagonist-centric network — expected)")

CHECK 1 — Edge count and weight distribution
  Total directed edges : 24
  Total utterances     : 390
  Weight — min=3  median=8  max=114

CHECK 2 — Reciprocity
  Reciprocity = 0.917
  NOTE: > 50% of edges have a return edge — dialogue is fairly mutual

CHECK 3 — Hub dominance
  Top node by betweenness: toy_story_1995_woody  (0.946)
  NOTE: single dominant hub (protagonist-centric network — expected)


## 7. Visualizations

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ── arc radii ──────────────────────────────────────────────────────────────
RAD_DEFAULT    = 0.20
RAD_RECIPROCAL = 0.35
edge_set = set(G.edges())
rad_map  = {}
for (u, v) in G.edges():
    rad_map[(u, v)] = RAD_RECIPROCAL if (v, u) in edge_set else RAD_DEFAULT

# ── sqrt width scaling ─────────────────────────────────────────────────────
max_w = max(G[u][v]['weight'] for u, v in G.edges())
def edge_width(w):
    return 1.0 + 9.0 * (w ** 0.5) / (max_w ** 0.5)

# ── Node color by gender ───────────────────────────────────────────────────
FEMALE_COLOR  = '#e05c8a'   # pink/rose
MALE_COLOR    = '#4a90d9'   # blue
UNKNOWN_COLOR = '#95a5a6'   # gray

def node_color(n):
    g = GENDER.get(n, 'Unknown')
    if g == 'F':   return FEMALE_COLOR
    if g == 'M':   return MALE_COLOR
    return UNKNOWN_COLOR

# ── Figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 13))
ax.set_facecolor('#f5f5f5'); fig.patch.set_facecolor('#f5f5f5')
pos = nx.spring_layout(G, seed=42, k=2.5, iterations=80)

nodes_list = list(G.nodes())
bw    = np.array([betweenness[n] for n in nodes_list])
bw_n  = (bw - bw.min()) / (bw.max() - bw.min() + 1e-9)
nsz   = 800 + bw_n * 3500
nsz_per_node = dict(zip(nodes_list, nsz))
nc    = [node_color(n) for n in nodes_list]

nx.draw_networkx_nodes(G, pos, node_size=nsz, node_color=nc, alpha=0.93, ax=ax)

for (u, v) in G.edges():
    d   = G[u][v]
    col = '#4a90d9' if (v, u) in edge_set else '#999999'
    nx.draw_networkx_edges(
        G, pos, edgelist=[(u, v)],
        edge_color=[col],
        width=[edge_width(d['weight'])],
        arrows=True, arrowstyle='-|>', arrowsize=14,
        connectionstyle=f'arc3,rad={rad_map[(u, v)]}',
        alpha=0.80, ax=ax,
    )

nx.draw_networkx_labels(G, pos, labels={n: id_to_name.get(n, n) for n in G.nodes()}, font_size=9, font_weight='bold', ax=ax)

ax.legend(handles=[
    Patch(facecolor=FEMALE_COLOR, label='Female character'),
    Patch(facecolor=MALE_COLOR,   label='Male character'),
    Line2D([0],[0],color='#4a90d9',lw=2.5,label='Reciprocal dialogue'),
    Line2D([0],[0],color='#999999',lw=2.5,label='One-directional dialogue'),
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#888888',markersize=14,label='Large node = high betweenness'),
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#888888',markersize=7, label='Small node = low betweenness'),
], loc='upper left', fontsize=8.5, framealpha=0.88)

ax.set_title(
    f"{FILM_TITLE} Dialogue Network\n"
    f"Directed addressee ties; node size = betweenness; color = gender; edge width = utterance count",
    fontsize=13, fontweight='bold', pad=16,
)
ax.axis('off')
plt.tight_layout()

# ── Edge weight labels ─────────────────────────────────────────────────────
fig.canvas.draw()
trans = ax.transData
inv   = ax.transData.inverted()
pt2px = fig.dpi / 72.0
pos_d = {n: trans.transform(pos[n]) for n in G.nodes()}

def node_radius_dcs(n):
    return np.sqrt(nsz_per_node[n] / np.pi) * pt2px

for (u, v) in G.edges():
    d      = G[u][v]
    w      = d['weight']
    rad    = rad_map[(u, v)]
    ux_d, uy_d = pos_d[u]
    vx_d, vy_d = pos_d[v]
    dx_d, dy_d = vx_d - ux_d, vy_d - uy_d
    dist_d = float(np.hypot(dx_d, dy_d))
    su = min(node_radius_dcs(u) / dist_d, 0.45)
    sv = min(node_radius_dcs(v) / dist_d, 0.45)
    ux2_d = ux_d + su * dx_d;  uy2_d = uy_d + su * dy_d
    vx2_d = vx_d - sv * dx_d;  vy2_d = vy_d - sv * dy_d
    dx2_d, dy2_d = vx2_d - ux2_d, vy2_d - uy2_d
    lx_d = (ux2_d + vx2_d) / 2 + (rad / 2) * dy2_d
    ly_d = (uy2_d + vy2_d) / 2 - (rad / 2) * dx2_d
    lx, ly = inv.transform([lx_d, ly_d])
    fsize  = 8  if w >= 8 else 7
    bold   = 'bold' if w >= 8 else 'normal'
    alpha  = 0.90 if w >= 8 else 0.75
    ax.text(lx, ly, str(w), fontsize=fsize, ha='center', va='center',
            color='#111111', fontweight=bold,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=alpha, ec='none'))

plt.savefig(PLOT_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved → {PLOT_PATH}')


Saved → ..\data\processed\toy_story_1995\figures\dialogue_network.png


C:\Users\majoa\AppData\Local\Temp\ipykernel_19460\3833362109.py:110: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── 7b. Centrality bar chart ───────────────────────────────────────────────
top_n = summary.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Most central characters in {FILM_TITLE}", fontweight="bold", fontsize=13)

# Betweenness
ax = axes[0]
bars = ax.barh(top_n["character"][::-1], top_n["betweenness"][::-1],
               color=plt.cm.Blues(0.6), edgecolor="white")
ax.set_xlabel("Weighted betweenness (higher = more bridging)")
ax.set_title("Dialogue bridges (top 10 by betweenness)", fontweight="bold")
for bar, v in zip(bars, top_n["betweenness"][::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", fontsize=8)

# In vs Out degree
ax = axes[1]
x  = np.arange(len(top_n))
w  = 0.35
ax.bar(x - w/2, top_n["in_degree"],  w, label="In-degree",  color="#4C72B0", alpha=0.85)
ax.bar(x + w/2, top_n["out_degree"], w, label="Out-degree", color="#DD8452", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(top_n["character"], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Number of distinct dialogue partners"); ax.set_title("Who speaks to whom? In-degree vs out-degree", fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(CENT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {CENT_PATH}")

Saved → ..\data\processed\toy_story_1995\figures\centrality_bar.png


C:\Users\majoa\AppData\Local\Temp\ipykernel_19460\3140386342.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save outputs

In [11]:
summary.to_csv(NODES_PATH, index=False)
agg.to_csv(EDGES_PATH,    index=False)

print(f"Saved node summary ({len(summary)} rows) → {NODES_PATH}")
print(f"Saved edge table   ({len(agg)} rows)     → {EDGES_PATH}")
print(f"\nFigures:")
print(f"  {PLOT_PATH}")
print(f"  {CENT_PATH}")

Saved node summary (12 rows) → ..\data\processed\toy_story_1995\network_node_summary.csv
Saved edge table   (24 rows)     → ..\data\processed\toy_story_1995\network_edges.csv

Figures:
  ..\data\processed\toy_story_1995\figures\dialogue_network.png
  ..\data\processed\toy_story_1995\figures\centrality_bar.png


## 9. Protagonist metrics

Compute three film-level metrics for the protagonist and save to `protagonist_metrics.csv`:

| Metric | Description |
|---|---|
| `ego_density` | Fraction of possible edges among the protagonist's alters that actually exist |
| `homophily_obs` | Share of utterances exchanged between same-gender characters |
| `homophily_expected` | Expected share under the configuration-model null (cast gender skew only) |
| `homophily_excess` | `obs − expected` — the part not explained by cast composition |

In [12]:
# ── 9a. Ego network density ────────────────────────────────────────────────
# Alters = all in- and out-neighbours of the protagonist (excluding self)
alters = (set(G.predecessors(PROTAGONIST)) | set(G.successors(PROTAGONIST)))
alters.discard(PROTAGONIST)

alter_edges  = [(u, v) for u, v in G.edges() if u in alters and v in alters]
n_alters     = len(alters)
max_possible = n_alters * (n_alters - 1)   # directed graph
ego_density  = len(alter_edges) / max_possible if max_possible > 0 else 0.0

print(f"Protagonist        : {PROTAGONIST_LABEL}")
print(f"Alters ({n_alters})        : {sorted(alters)}")
print(f"Edges among alters : {len(alter_edges)} / {max_possible} possible")
print(f"Ego network density: {ego_density:.4f}")

# ── 9b. Null-corrected homophily ───────────────────────────────────────────
mm = mf = fm = ff = 0
for u, v, d in G.edges(data=True):
    w  = d["weight"]
    gu = GENDER.get(u)
    gv = GENDER.get(v)
    if gu is None or gv is None:
        continue
    if   gu == "M" and gv == "M": mm += w
    elif gu == "M" and gv == "F": mf += w
    elif gu == "F" and gv == "M": fm += w
    else:                          ff += w

total              = mm + mf + fm + ff
p_out_M            = (mm + mf) / total
p_out_F            = (fm + ff) / total
p_in_M             = (mm + fm) / total
p_in_F             = (mf + ff) / total
homophily_obs      = (mm + ff) / total
homophily_expected = p_out_M * p_in_M + p_out_F * p_in_F
homophily_excess   = homophily_obs - homophily_expected

print(f"\nGender pair counts (utterances):")
print(f"  M→M={mm}  M→F={mf}  F→M={fm}  F→F={ff}  total={total}")
print(f"\nHomophily observed : {homophily_obs:.4f}")
print(f"Homophily expected : {homophily_expected:.4f}  (configuration-model null)")
print(f"Excess homophily   : {homophily_excess:+.4f}")

# ── 9c. Save ───────────────────────────────────────────────────────────────
protagonist_row = summary[summary["character"] == PROTAGONIST].iloc[0]

metrics = pd.DataFrame([{
    "film_id":            FILM_ID,
    "protagonist":        PROTAGONIST_LABEL,
    "betweenness":        protagonist_row["betweenness"],
    "degree_centrality":  protagonist_row["degree_centrality"],
    "in_degree":          int(protagonist_row["in_degree"]),
    "out_degree":         int(protagonist_row["out_degree"]),
    "n_alters":           n_alters,
    "alter_edges":        len(alter_edges),
    "ego_density":        round(ego_density, 4),
    "homophily_obs":      round(homophily_obs, 4),
    "homophily_expected": round(homophily_expected, 4),
    "homophily_excess":   round(homophily_excess, 4),
}])

metrics.to_csv(PROTAGONIST_METRICS_PATH, index=False)
print(f"\nSaved → {PROTAGONIST_METRICS_PATH}")
print(metrics.T.to_string(header=False))


Protagonist        : Woody
Alters (9)        : ['toy_story_1995_andy', 'toy_story_1995_bo_peep', 'toy_story_1995_buzz', 'toy_story_1995_hamm', 'toy_story_1995_mr_potato_head', 'toy_story_1995_rex', 'toy_story_1995_sargent', 'toy_story_1995_sid', 'toy_story_1995_slinky']
Edges among alters : 3 / 72 possible
Ego network density: 0.0417

Gender pair counts (utterances):
  M→M=310  M→F=36  F→M=44  F→F=0  total=390

Homophily observed : 0.7949
Homophily expected : 0.8157  (configuration-model null)
Excess homophily   : -0.0208

Saved → ..\data\processed\toy_story_1995\protagonist_metrics.csv
film_id             toy_story_1995
protagonist                  Woody
betweenness                 0.9455
degree_centrality           1.5455
in_degree                        9
out_degree                       8
n_alters                         9
alter_edges                      3
ego_density                 0.0417
homophily_obs               0.7949
homophily_expected          0.8157
homophily_excess     

## 10. Configuration-model null for homophily (inference)

Following the advising email: turn homophily from a **descriptive** number into an
**inferential** one. We hold each character's **in- and out-degree fixed** (the
configuration model) and **randomly rewire** who talks to whom via degree-preserving
directed edge swaps. Gender composition is held fixed automatically — every node keeps
its own identity (and gender); only the *targets* of dialogue are randomized.

Repeating this `N_NULL_SAMPLES` times gives a **null distribution** of homophily under
*"each character talks to the same number of people, but at random."* The test is
**edge-based / unweighted** (each directed tie counts once), matching the advisor's
guidance to *keep the degree and forget the weights* for the null.

We then report where the observed value falls in that distribution: its **z-score**,
**percentile**, and a one-sided empirical **p-value**.

**Two measures, same null.** First the **film-level** homophily above. Then — following the advisor's emphasis on looking at the *lead's friends* — the **protagonist's neighbour same-sex proportion**: of the characters the lead actually talks with, what share are her own gender? Benchmarking against the configuration model (not the raw cast share) is essential here: by the *friendship paradox*, a popular character is over-represented in everyone's neighbourhood, so the random baseline is **not** simply the cast's gender split.


In [13]:
import random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── 11a. Observed edge-based homophily (unweighted, gendered nodes only) ─────
base_edges = [
    (u, v) for u, v in G.edges()
    if u != v and GENDER.get(u) is not None and GENDER.get(v) is not None
]
E = len(base_edges)
hom_edge_count = sum(1 for u, v in base_edges if GENDER[u] == GENDER[v])
hom_edge_obs = hom_edge_count / E

# ── 11b. Degree-preserving directed edge-swap ensemble ──────────────────────
# A swap takes ties a->b and c->d to a->d and c->b, leaving every node's
# in-degree and out-degree unchanged. Rejecting self-loops and duplicates
# keeps each randomized network a simple directed graph.
def edge_swap_sample(edges, n_swaps, rng):
    edges = list(edges)
    edge_set = set(edges)
    m = len(edges)
    done = attempts = 0
    max_attempts = n_swaps * 20
    while done < n_swaps and attempts < max_attempts:
        attempts += 1
        i, j = rng.randrange(m), rng.randrange(m)
        if i == j:
            continue
        a, b = edges[i]
        c, d = edges[j]
        if a == d or c == b:                       # self-loop
            continue
        if (a, d) in edge_set or (c, b) in edge_set:  # duplicate edge
            continue
        edge_set.discard((a, b)); edge_set.discard((c, d))
        edge_set.add((a, d));     edge_set.add((c, b))
        edges[i] = (a, d);        edges[j] = (c, b)
        done += 1
    return edges

rng        = random.Random(NULL_SEED)
swaps      = 10 * E                       # mixing per sample
null_vals  = np.empty(N_NULL_SAMPLES)
for s in range(N_NULL_SAMPLES):
    sample = edge_swap_sample(base_edges, swaps, rng)
    null_vals[s] = sum(1 for u, v in sample if GENDER[u] == GENDER[v]) / E

# ── 11c. Inference: z-score, percentile, one-sided p-value ──────────────────
null_mean = null_vals.mean()
null_std  = null_vals.std(ddof=1)
z_score   = (hom_edge_obs - null_mean) / null_std if null_std > 0 else float("nan")
percentile = (null_vals < hom_edge_obs).mean() * 100
# one-sided: P(null >= observed), with +1 smoothing
p_upper = (np.sum(null_vals >= hom_edge_obs) + 1) / (N_NULL_SAMPLES + 1)
p_lower = (np.sum(null_vals <= hom_edge_obs) + 1) / (N_NULL_SAMPLES + 1)

print(f"Edges in null (gendered) : {E}")
print(f"Observed homophily (edge): {hom_edge_obs:.4f}")
print(f"Null mean ± sd           : {null_mean:.4f} ± {null_std:.4f}")
print(f"z-score                  : {z_score:+.3f}")
print(f"Percentile in null       : {percentile:.1f}%")
print(f"p (more same-gender)     : {p_upper:.4f}")
print(f"p (less same-gender)     : {p_lower:.4f}")

# ── 11d. Plot null distribution with observed value ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))

# Homophily can only equal k/E (E = # gendered ties), so on a small graph the
# null distribution is inherently discrete. Draw one bar per achievable value
# rather than 30 arbitrary bins (which would leave most bins empty).
counts    = np.rint(null_vals * E).astype(int)
lo, hi    = counts.min(), counts.max()
bin_edges = (np.arange(lo, hi + 2) - 0.5) / E
ax.hist(null_vals, bins=bin_edges, color="#9ecae1", edgecolor="white", alpha=0.9)
ax.set_xticks(np.arange(lo, hi + 1) / E)
ax.set_xticklabels([f"{k}/{E}" for k in range(lo, hi + 1)], fontsize=8)
ax.axvline(null_mean, color="#3182bd", linestyle="--", linewidth=1.5,
           label=f"null mean = {null_mean:.3f}")
ax.axvline(hom_edge_obs, color="#de2d26", linewidth=2.5,
           label=f"observed = {hom_edge_obs:.3f}")
ax.set_xlabel(f"Same-gender directed dialogue ties, out of {E} gender-coded ties")
ax.set_ylabel("Randomized networks")
ax.set_title(f"Whole-cast same-gender dialogue ties vs. random rewiring — {FILM_TITLE}\n"
             f"observed = {hom_edge_count}/{E} ({hom_edge_obs:.1%}); null mean = {null_mean:.1%}; p = {p_upper:.3f} (N = {N_NULL_SAMPLES})",
             fontweight="bold", fontsize=11)
ax.legend()
fig.tight_layout()
fig.savefig(HOMOPHILY_NULL_FIG, dpi=150, bbox_inches="tight")
plt.close(fig)

# ── 10f. Protagonist's neighbour same-sex proportion vs. the same null ──────
# Advisor's lead-centric measure: of the people the protagonist talks with, what
# fraction share her gender? Benchmarked against the same degree-preserving null,
# which controls for popularity (friendship paradox) — so the baseline is NOT the
# raw cast share.
prot_gender = GENDER.get(PROTAGONIST)

def neighbour_samesex_frac(edges, ego, ego_gender):
    nbrs = set()
    for u, v in edges:
        if u == ego:   nbrs.add(v)
        elif v == ego: nbrs.add(u)
    nbrs = [n for n in nbrs if GENDER.get(n) is not None]
    if not nbrs:
        return float("nan")
    return sum(1 for n in nbrs if GENDER[n] == ego_gender) / len(nbrs)

prot_ss_obs = neighbour_samesex_frac(base_edges, PROTAGONIST, prot_gender)

rng_ss  = random.Random(NULL_SEED + 1)
ss_null = np.empty(N_NULL_SAMPLES)
for s in range(N_NULL_SAMPLES):
    ss_null[s] = neighbour_samesex_frac(
        edge_swap_sample(base_edges, swaps, rng_ss), PROTAGONIST, prot_gender)

ss_mean = np.nanmean(ss_null)
ss_std  = np.nanstd(ss_null, ddof=1)
ss_z    = (prot_ss_obs - ss_mean) / ss_std if ss_std > 0 else float("nan")
ss_pct  = np.mean(ss_null < prot_ss_obs) * 100
ss_p_hi = (np.sum(ss_null >= prot_ss_obs) + 1) / (N_NULL_SAMPLES + 1)
ss_p_lo = (np.sum(ss_null <= prot_ss_obs) + 1) / (N_NULL_SAMPLES + 1)

print(f"\n{PROTAGONIST_LABEL} ({prot_gender}) neighbour same-sex proportion")
print(f"  observed          : {prot_ss_obs:.4f}")
print(f"  null mean +/- sd  : {ss_mean:.4f} +/- {ss_std:.4f}")
print(f"  z-score           : {ss_z:+.3f}")
print(f"  percentile        : {ss_pct:.1f}%")
print(f"  p (more same-sex) : {ss_p_hi:.4f}")
print(f"  p (less same-sex) : {ss_p_lo:.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(ss_null[~np.isnan(ss_null)], bins=20, color="#c994c7", edgecolor="white", alpha=0.9)
ax.axvline(ss_mean, color="#8856a7", linestyle="--", linewidth=1.5,
           label=f"null mean = {ss_mean:.3f}")
ax.axvline(prot_ss_obs, color="#de2d26", linewidth=2.5,
           label=f"observed = {prot_ss_obs:.3f}")
ax.set_xlabel(f"Share of {PROTAGONIST_LABEL}'s dialogue partners who are same-gender ({prot_gender})")
ax.set_ylabel("Randomized networks")
ax.set_title(f"{PROTAGONIST_LABEL}'s same-gender dialogue partners vs. random rewiring — {FILM_TITLE}\n"
             f"observed = {prot_ss_obs:.1%}; null mean = {ss_mean:.1%}; z = {ss_z:+.2f}; p = {ss_p_hi:.3f} (N = {N_NULL_SAMPLES})",
             fontweight="bold", fontsize=11)
ax.legend()
fig.tight_layout()
fig.savefig(PROT_SAMESEX_FIG, dpi=150, bbox_inches="tight")
plt.close(fig)

# ── 11e. Save ───────────────────────────────────────────────────────────────
null_test = pd.DataFrame([{
    "film_id":            FILM_ID,
    "protagonist":        PROTAGONIST_LABEL,
    "n_edges":            E,
    "n_null_samples":     N_NULL_SAMPLES,
    "homophily_obs_edge": round(hom_edge_obs, 4),
    "null_mean":          round(null_mean, 4),
    "null_std":           round(null_std, 4),
    "z_score":            round(z_score, 3),
    "percentile":         round(percentile, 1),
    "p_more_same_gender": round(p_upper, 4),
    "p_less_same_gender": round(p_lower, 4),
    "protag_samesex_obs":    round(prot_ss_obs, 4),
    "protag_samesex_null":   round(ss_mean, 4),
    "protag_samesex_z":      round(ss_z, 3),
    "protag_samesex_pct":    round(ss_pct, 1),
    "protag_samesex_p_more": round(ss_p_hi, 4),
    "protag_samesex_p_less": round(ss_p_lo, 4),
}])
null_test.to_csv(HOMOPHILY_NULL_PATH, index=False)
print(f"\nSaved → {HOMOPHILY_NULL_PATH}")
print(f"Figure → {HOMOPHILY_NULL_FIG}")
print(null_test.T.to_string(header=False))


Edges in null (gendered) : 24
Observed homophily (edge): 0.7500
Null mean ± sd           : 0.7551 ± 0.0204
z-score                  : -0.252
Percentile in null       : 0.0%
p (more same-gender)     : 1.0000
p (less same-gender)     : 0.9395

Woody (M) neighbour same-sex proportion
  observed          : 0.8889
  null mean +/- sd  : 0.7346 +/- 0.0365
  z-score           : +4.225
  percentile        : 99.5%
  p (more same-sex) : 0.0055
  p (less same-sex) : 1.0000

Saved → ..\data\processed\toy_story_1995\homophily_null.csv
Figure → ..\data\processed\toy_story_1995\figures\homophily_null.png
film_id                toy_story_1995
protagonist                     Woody
n_edges                            24
n_null_samples                   2000
homophily_obs_edge               0.75
null_mean                      0.7551
null_std                       0.0204
z_score                        -0.252
percentile                        0.0
p_more_same_gender                1.0
p_less_same_gender      

## 11. Configuration-model null for betweenness (inference)

The same degree-preserving rewiring, now applied to **betweenness centrality** — the
advisor explicitly asked for hypothesis testing on betweenness. Question: is a character's
bridging role larger than expected if everyone kept their number of conversation partners
but rewired at random?

We rewire the **full** directed network (all characters — bridging is structural, not
gender-specific), recompute **weighted** betweenness each sample (`distance = 1/utterance_count`,
the same definition that selects the rung-3 keystone in §12, so the inferential test and the
keystone selection are on the same footing), and locate every character's observed value in its
own null distribution. The randomization engine is unchanged — only the statistic measured on top
is weighted: the observed weight multiset is permuted onto the rewired ties, so every node's
in/out-degree is still preserved exactly. The headline is the protagonist; the per-character table
flags any node — often a *secondary* one — whose bridging role is significantly above chance (the
"Horatio / Anna" structural-importance finding).

Reuses `edge_swap_sample` from §11.

In [14]:
# ── 12a. Observed WEIGHTED betweenness, full graph ──────────────────────────
# Match the keystone definition (§12, rung 3): distance = 1/utterance_count, so
# heavier dialogue ties are SHORTER paths. This makes the null test the SAME
# statistic that selects the rung-3 keystone.
nodes        = list(G.nodes())
full_edges   = [(u, v) for u, v in G.edges() if u != v]
weights      = [G[u][v]["weight"] for (u, v) in full_edges]   # observed weight multiset
for u, v in full_edges:
    G[u][v]["distance"] = 1.0 / G[u][v]["weight"]
obs_bet      = nx.betweenness_centrality(G, weight="distance", normalized=True)

# ── 12b. Null ensemble: rewire full degree sequence, recompute betweenness ──
# Randomization engine is UNCHANGED (degree-preserving edge swap). To evaluate the
# weighted statistic on each sample, the observed weight multiset is permuted onto
# the rewired ties (same rng — no new randomness source) and converted to distances.
# Every node's in/out-degree is preserved exactly; only the measured statistic is weighted.
rng       = random.Random(NULL_SEED)
swaps     = 10 * len(full_edges)
null_bet  = {n: np.empty(N_NULL_SAMPLES) for n in nodes}
for s in range(N_NULL_SAMPLES):
    sample = edge_swap_sample(full_edges, swaps, rng)
    perm   = weights[:]
    rng.shuffle(perm)                          # permute observed weights onto rewired ties
    R = nx.DiGraph()
    R.add_nodes_from(nodes)                     # keep isolates / identities intact
    R.add_edges_from(
        (u, v, {"distance": 1.0 / w}) for (u, v), w in zip(sample, perm)
    )
    bc = nx.betweenness_centrality(R, weight="distance", normalized=True)
    for n in nodes:
        null_bet[n][s] = bc[n]

# ── 12c. Per-character inference ────────────────────────────────────────────
rows = []
for n in nodes:
    nv  = null_bet[n]
    m   = nv.mean()
    sd  = nv.std(ddof=1)
    obs = obs_bet[n]
    z   = (obs - m) / sd if sd > 0 else float("nan")
    rows.append({
        "character":   n,
        "gender":      GENDER.get(n),
        "betw_obs":    round(obs, 4),
        "null_mean":   round(m, 4),
        "null_std":    round(sd, 4),
        "z_score":     round(z, 3),
        "percentile":  round((nv < obs).mean() * 100, 1),
        "p_high":      round((np.sum(nv >= obs) + 1) / (N_NULL_SAMPLES + 1), 4),
    })

betw_null = (
    pd.DataFrame(rows)
      .sort_values("betw_obs", ascending=False)
      .reset_index(drop=True)
)
betw_null.insert(0, "film_id", FILM_ID)

print(f"Edges rewired (full graph): {len(full_edges)}   |   N = {N_NULL_SAMPLES}")
print("\nWeighted betweenness vs. configuration-model null (sorted by observed):")
print(betw_null.drop(columns="film_id").to_string(index=False))

prow = betw_null[betw_null["character"] == PROTAGONIST].iloc[0]
print(f"\nProtagonist ({PROTAGONIST_LABEL}): betw_obs={prow['betw_obs']:.4f}  "
      f"null_mean={prow['null_mean']:.4f}  z={prow['z_score']:+.3f}  "
      f"percentile={prow['percentile']:.1f}%  p_high={prow['p_high']:.4f}")

# ── 12d. Plot the protagonist's null distribution ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
pv = null_bet[PROTAGONIST]
ax.hist(pv, bins=30, color="#a1d99b", edgecolor="white", alpha=0.9)
ax.axvline(pv.mean(), color="#31a354", linestyle="--", linewidth=1.5,
           label=f"null mean = {pv.mean():.3f}")
ax.axvline(obs_bet[PROTAGONIST], color="#de2d26", linewidth=2.5,
           label=f"observed = {obs_bet[PROTAGONIST]:.3f}")
ax.set_xlabel("Weighted betweenness centrality (distance = 1/utterances)")
ax.set_ylabel("Randomized networks")
ax.set_title(f"Is {PROTAGONIST_LABEL} more of a dialogue bridge than expected? — {FILM_TITLE}\n"
             f"observed = {obs_bet[PROTAGONIST]:.3f}; null mean = {pv.mean():.3f}; z = {prow['z_score']:+.2f}; p = {prow['p_high']:.3f} (N = {N_NULL_SAMPLES})",
             fontweight="bold", fontsize=11)
ax.legend()
fig.tight_layout()
fig.savefig(BETWEENNESS_NULL_FIG, dpi=150, bbox_inches="tight")
plt.close(fig)

# ── 12e. Save ───────────────────────────────────────────────────────────────
betw_null.to_csv(BETWEENNESS_NULL_PATH, index=False)
print(f"\nSaved → {BETWEENNESS_NULL_PATH}")
print(f"Figure → {BETWEENNESS_NULL_FIG}")

Edges rewired (full graph): 24   |   N = 2000

Weighted betweenness vs. configuration-model null (sorted by observed):
                    character gender  betw_obs  null_mean  null_std  z_score  percentile  p_high
         toy_story_1995_woody      M    0.9455     0.8964    0.0347    1.412        95.5  0.0455
          toy_story_1995_andy      M    0.1818     0.1171    0.0819    0.790        81.8  0.1819
toy_story_1995_mr_potato_head      M    0.1818     0.1190    0.0819    0.767        80.8  0.1919
           toy_story_1995_sid      M    0.1818     0.1194    0.0824    0.758        81.4  0.1869
     toy_story_1995_mrs_davis      F    0.0000     0.0239    0.0494   -0.482         0.0  1.0000
          toy_story_1995_buzz      M    0.0000     0.0553    0.0660   -0.838         0.0  1.0000
        toy_story_1995_hannah      F    0.0000     0.0229    0.0472   -0.485         0.0  1.0000
        toy_story_1995_slinky      M    0.0000     0.0243    0.0498   -0.488         0.0  1.0000
        

## 12. Methodological-ladder keystones (Moretti)

Three rungs of network construction yield three candidate keystones. The
selection rule is the same at every rung — Moretti's "highest-betweenness
non-protagonist" — only the underlying network changes. The ladder adds one
piece of information per rung:

| Rung | Network | What's added | Method cost |
|---|---|---|---|
| 1 | **Co-occurrence** (undirected, unweighted) | who shares scenes | scene list only — Moretti baseline |
| 2 | Addressee, **directed + unweighted** | + direction (who speaks *to* whom) | **requires LLM addressee tagging** |
| 3 | **Addressee, directed + weighted — PRIMARY** | + weight (utterance count per tie) | full LLM output |

**The LLM's unique affordance is added at rung 2 (direction), not rung 3.**
Rung 3 then refines rung 2 by weighting ties by dialogue volume. Rungs 2 and 3
operate on the *same* directed addressee graph `G_wD`; the difference is purely
whether betweenness is computed unweighted (rung 2) or with `distance = 1/w`
(rung 3).

The rung-3 keystone is the headline — it drives the removal / fragmentation
test below (Moretti's "Horatio" question: how badly does the network shatter
when the structural bridge is taken out?). Rungs 1 and 2 are saved alongside
so notebook 09 can trace transitions across the ladder. If a rung's network
has all non-protagonist betweenness tied at zero (e.g. a near-complete
co-occurrence graph with no real bridge), that rung returns `None` rather
than the iteration-order artefact `max()` would otherwise produce.

After picking the rung-3 keystone, we also remove **each same-gender alter of
the protagonist** one at a time to test whether the lead's same-gender
embeddedness rests on one load-bearing tie or a distributed community.

In [15]:
# ── 12a. Methodological-ladder keystone selection ────────────────────────
# Same rule (highest-betweenness non-protagonist) at every rung; only the
# underlying network changes. The ladder adds ONE piece of information per rung:
#
#   Rung 1  Co-occurrence (undirected, unweighted)             Moretti baseline
#   Rung 2  Addressee, directed + unweighted   ← LLM affordance: + DIRECTION
#   Rung 3  Addressee, directed + weighted   ← PRIMARY             + WEIGHT
#
# Rungs 2 and 3 share the same directed addressee graph G_wD; the difference
# is whether betweenness is computed unweighted (rung 2) or weighted (rung 3).

# --- Rung 1: co-occurrence from raw utterances (no LLM addressee needed) ---
df_cooc = df.copy()
df_cooc["character_id"] = df_cooc["character_id"].replace(ALIAS)
df_cooc = df_cooc.dropna(subset=["scene_id", "character_id"])
_scene_speakers = (df_cooc.groupby("scene_id")["character_id"]
                       .apply(lambda s: sorted(set(s))))
_cooc_pairs = {}
for _chars in _scene_speakers:
    if len(_chars) < 2:
        continue
    for _i in range(len(_chars)):
        for _j in range(_i + 1, len(_chars)):
            _key = frozenset({_chars[_i], _chars[_j]})
            _cooc_pairs[_key] = _cooc_pairs.get(_key, 0) + 1
_cooc_pairs = {k: v for k, v in _cooc_pairs.items() if v >= MIN_EDGE_COUNT}
G_cooc = nx.Graph()
for _k in _cooc_pairs:
    _a, _b = list(_k)
    G_cooc.add_edge(_a, _b)                                       # unweighted

# --- Rungs 2 and 3 share the directed addressee graph -------------------
# Rung 2 is the SAME graph with unweighted betweenness; rung 3 weights by
# distance = 1/utterance_count (so heavier ties = shorter paths).
G_wD = G.copy()
for a, b, d in G_wD.edges(data=True):
    d["distance"] = 1.0 / d["weight"]

# --- Keystone at each rung -----------------------------------------------
def _pick_keystone(graph, protag, weight=None):
    """Highest-betweenness non-protagonist; returns None if all-tied at zero."""
    if protag not in graph or graph.number_of_edges() == 0:
        return None
    btw = nx.betweenness_centrality(graph, weight=weight, normalized=True)
    non_lead = [(n, v) for n, v in btw.items() if n != protag]
    if not non_lead:
        return None
    if max(v for _, v in non_lead) == 0:
        return None                                  # no real bridge at this rung
    return max(non_lead, key=lambda x: x[1])[0]

ks_rung1 = _pick_keystone(G_cooc, PROTAGONIST, weight=None)         # cooc, unweighted
ks_rung2 = _pick_keystone(G_wD,   PROTAGONIST, weight=None)         # directed, unweighted
ks_rung3 = _pick_keystone(G_wD,   PROTAGONIST, weight="distance")   # directed + weighted (PRIMARY)

print(f"{'Rung':<22}{'Keystone':<28}{'Gender'}")
print("-" * 60)
for _lbl, _n in [("rung1_cooc",     ks_rung1),
                 ("rung2_directed", ks_rung2),
                 ("rung3_dir_wgt",  ks_rung3)]:
    _star = "  <-- PRIMARY" if _lbl == "rung3_dir_wgt" else ""
    _nm   = id_to_name.get(_n, _n) if _n else "(none - all-tied / no bridge)"
    _g    = GENDER.get(_n, "-") if _n else "-"
    print(f"{_lbl:<22}{str(_nm):<28}{_g}{_star}")

changes_at_rung2 = (ks_rung1 is not None and ks_rung2 is not None and ks_rung1 != ks_rung2)
changes_at_rung3 = (ks_rung2 is not None and ks_rung3 is not None and ks_rung2 != ks_rung3)
print(f"\nKeystone changes rung 1 -> rung 2 (adding direction): {changes_at_rung2}")
print(f"Keystone changes rung 2 -> rung 3 (adding weight):    {changes_at_rung3}")

# Wire the rung-3 (PRIMARY) keystone into the existing removal analysis.
# REMOVAL_TARGET in §0 can override (for sensitivity checks); otherwise rung 3.
keystone = REMOVAL_TARGET if REMOVAL_TARGET else ks_rung3
if keystone is None:
    raise RuntimeError(
        "No rung-3 keystone found. Set REMOVAL_TARGET in §0 to override, "
        "or check the directed addressee network for this film."
    )
keystone_betw = nx.betweenness_centrality(G_wD, weight="distance",
                                          normalized=True).get(keystone, float("nan"))

print(f"\nProtagonist (kept)   : {PROTAGONIST_LABEL}")
print(f"Keystone removed     : {id_to_name.get(keystone, keystone)}  "
      f"(weighted directed betweenness = {keystone_betw:.4f})")
print()

# ── 12b. Remove the keystone → measure fragmentation ─────────────────────
G_removed = G.copy()
G_removed.remove_node(keystone)

components        = list(nx.weakly_connected_components(G_removed))
components_sorted = sorted(components, key=len, reverse=True)
n_components      = len(components_sorted)
largest_size      = len(components_sorted[0]) if components_sorted else 0
n_isolated        = sum(1 for c in components_sorted if len(c) == 1)
original_n        = G.number_of_nodes()

print(f"Original network         : {original_n} nodes")
print(f"After removing {id_to_name.get(keystone, keystone)} : "
      f"{G_removed.number_of_nodes()} nodes")
print(f"  Weakly connected components : {n_components}")
print(f"  Largest component size      : {largest_size}  "
      f"({largest_size/(original_n-1)*100:.0f}% of remaining nodes)")
print(f"  Isolated nodes              : {n_isolated}")
print()
for i, comp in enumerate(components_sorted):
    print(f"  Component {i+1} ({len(comp)} nodes): {sorted(comp)}")

# ── 12c. Remove each SAME-GENDER alter of the protagonist → ego density ──
lead_gender       = GENDER.get(PROTAGONIST)
samegender_alters = [n for n in alters if GENDER.get(n) == lead_gender]
print()
print(f"Same-gender ({lead_gender}) alters of {PROTAGONIST_LABEL}: {samegender_alters}")
print(f"Baseline ego density : {ego_density:.4f}  ({len(alter_edges)} / {max_possible} edges)")
print()
print(f"{'Alter removed':<20}  {'Edges':<6}  {'Max':<6}  {'Density':<8}  {'Drop'}")
print("-" * 58)
for alter in sorted(samegender_alters):
    remaining = [n for n in alters if n != alter]
    n_r     = len(remaining)
    edges_r = [(u, v) for u, v in G.edges() if u in remaining and v in remaining]
    max_r   = n_r * (n_r - 1)
    dens_r  = len(edges_r) / max_r if max_r > 0 else 0.0
    drop    = ego_density - dens_r
    print(f"{alter:<20}  {len(edges_r):<6}  {max_r:<6}  {dens_r:<8.4f}  {drop:+.4f}")

# ── 12d. Save metrics: three-rung keystone + removal stats ───────────────
metrics_df = pd.read_csv(PROTAGONIST_METRICS_PATH)

# Three-rung keystones (rung 3 is PRIMARY)
metrics_df["keystone_rung1"]            = id_to_name.get(ks_rung1, ks_rung1) if ks_rung1 else None
metrics_df["keystone_rung1_gender"]     = GENDER.get(ks_rung1) if ks_rung1 else None
metrics_df["keystone_rung2"]            = id_to_name.get(ks_rung2, ks_rung2) if ks_rung2 else None
metrics_df["keystone_rung2_gender"]     = GENDER.get(ks_rung2) if ks_rung2 else None
metrics_df["keystone_rung3"]            = id_to_name.get(ks_rung3, ks_rung3) if ks_rung3 else None
metrics_df["keystone_rung3_gender"]     = GENDER.get(ks_rung3) if ks_rung3 else None
metrics_df["keystone_changes_at_rung2"] = int(changes_at_rung2)
metrics_df["keystone_changes_at_rung3"] = int(changes_at_rung3)

# Primary-keystone aliases (legacy column names point at rung 3)
metrics_df["keystone_removed"]          = metrics_df["keystone_rung3"]
metrics_df["keystone_gender"]           = metrics_df["keystone_rung3_gender"]
metrics_df["keystone_betweenness"]      = round(keystone_betw, 4) if keystone_betw == keystone_betw else None
metrics_df["components_after_removal"]  = n_components
metrics_df["largest_component_size"]    = largest_size
metrics_df["n_isolated_after_removal"]  = n_isolated
metrics_df["lead_gender"]               = lead_gender
metrics_df["n_samegender_alters"]       = len(samegender_alters)

metrics_df.to_csv(PROTAGONIST_METRICS_PATH, index=False)
print()
print(f"Saved removal metrics → {PROTAGONIST_METRICS_PATH}")

Rung                  Keystone                    Gender
------------------------------------------------------------
rung1_cooc            Andy                        M
rung2_directed        Andy                        M
rung3_dir_wgt         Andy                        M  <-- PRIMARY

Keystone changes rung 1 -> rung 2 (adding direction): False
Keystone changes rung 2 -> rung 3 (adding weight):    False

Protagonist (kept)   : Woody
Keystone removed     : Andy  (weighted directed betweenness = 0.1818)

Original network         : 12 nodes
After removing Andy : 11 nodes
  Weakly connected components : 2
  Largest component size      : 10  (91% of remaining nodes)
  Isolated nodes              : 1

  Component 1 (10 nodes): ['toy_story_1995_bo_peep', 'toy_story_1995_buzz', 'toy_story_1995_hamm', 'toy_story_1995_hannah', 'toy_story_1995_mr_potato_head', 'toy_story_1995_rex', 'toy_story_1995_sargent', 'toy_story_1995_sid', 'toy_story_1995_slinky', 'toy_story_1995_woody']
  Component 2 (1 n

In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

REMOVAL_PLOT_PATH = FIGURES_DIR / 'removal_analysis.png'
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.patch.set_facecolor('#f5f5f5')

# ── shared layout ─────────────────────────────────────────────────────────
layout_seed = nx.spring_layout(G, seed=42, k=2.5, iterations=80)
# layout for removed graph: keep same positions (drop protagonist node)
layout_removed = {n: layout_seed[n] for n in G_removed.nodes()}

def node_sizes(graph, base=600, scale=2800):
    bw = nx.betweenness_centrality(graph, normalized=True, weight='distance')
    vals = [bw[n] for n in graph.nodes()]
    mn, mx = min(vals), max(vals)
    return {n: base + scale * (bw[n]-mn)/(mx-mn+1e-9) for n in graph.nodes()}

# ── Panel 1: Original network ──────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#f5f5f5')
sz1 = node_sizes(G)
def _node_color(n):
    if n == PROTAGONIST: return '#e74c3c'   # lead (kept)
    if n == keystone:    return '#e67e22'   # keystone (removed in panel 2)
    return '#4a90d9'
node_colors1 = [_node_color(n) for n in G.nodes()]
nx.draw_networkx_nodes(G, layout_seed, node_size=[sz1[n] for n in G.nodes()],
                       node_color=node_colors1, alpha=0.90, ax=ax1)
nx.draw_networkx_edges(G, layout_seed, edge_color='#888888', width=1.2,
                       arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.15', alpha=0.60, ax=ax1)
nx.draw_networkx_labels(G, layout_seed, labels={n: id_to_name.get(n, n) for n in G.nodes()}, font_size=8, font_weight='bold', ax=ax1)
ax1.set_title(f'Before removal\n{G.number_of_nodes()} nodes · {G.number_of_edges()} edges',
              fontsize=11, fontweight='bold')
ax1.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label=f'Lead: {PROTAGONIST_LABEL} (kept)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e67e22', markersize=10, label=f'Keystone: {id_to_name.get(keystone, keystone)} (removed)'),
], fontsize=7.5, loc='lower left', framealpha=0.85)
ax1.axis('off')

# ── Panel 2: After removing the keystone ──────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#f5f5f5')
component_colors = ['#2ecc71', '#e67e22', '#9b59b6', '#1abc9c', '#e74c3c']
node_color2 = {}
for ci, comp in enumerate(components_sorted):
    for n in comp:
        node_color2[n] = component_colors[ci % len(component_colors)]
sz2 = node_sizes(G_removed)
nx.draw_networkx_nodes(G_removed, layout_removed,
                       node_size=[sz2[n] for n in G_removed.nodes()],
                       node_color=[node_color2[n] for n in G_removed.nodes()],
                       alpha=0.90, ax=ax2)
nx.draw_networkx_edges(G_removed, layout_removed, edge_color='#888888', width=1.2,
                       arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.15', alpha=0.60, ax=ax2)
nx.draw_networkx_labels(G_removed, layout_removed, labels={n: id_to_name.get(n, n) for n in G_removed.nodes()}, font_size=8, font_weight='bold', ax=ax2)
comp_patches = [mpatches.Patch(color=component_colors[i % len(component_colors)],
                label=f'Component {i+1} ({len(c)} nodes)')
                for i, c in enumerate(components_sorted)]
ax2.legend(handles=comp_patches, fontsize=7.5, loc='lower left', framealpha=0.85)
ax2.set_title(f'After removing {id_to_name.get(keystone, keystone)}\n{n_components} components · {n_isolated} isolated node(s)',
              fontsize=11, fontweight='bold')
ax2.axis('off')

# ── Panel 3: Same-gender alter removal → ego density ──────────────────────
ax3 = axes[2]
ax3.set_facecolor('#f5f5f5')
alter_names, densities = [], []
for alter in sorted(samegender_alters):
    remaining = [n for n in alters if n != alter]
    n_r     = len(remaining)
    edges_r = [(u, v) for u, v in G.edges() if u in remaining and v in remaining]
    max_r   = n_r * (n_r - 1)
    dens_r  = len(edges_r) / max_r if max_r > 0 else 0.0
    alter_names.append(alter)
    densities.append(dens_r)

labels  = ['Baseline'] + alter_names
values  = [ego_density] + densities
colors  = ['#4a90d9'] + ['#e74c3c' if v < ego_density else '#2ecc71' for v in densities]
bars = ax3.barh(labels[::-1], values[::-1], color=colors[::-1],
                edgecolor='white', height=0.5)
ax3.axvline(ego_density, color='#4a90d9', linestyle='--', linewidth=1.5, alpha=0.7)
for bar, val in zip(bars, values[::-1]):
    ax3.text(val + 0.003, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax3.set_xlabel('Ego network density', fontsize=10)
ax3.set_title(f"Lead ego density after removing\neach same-gender ({lead_gender}) alter",
              fontsize=11, fontweight='bold')
ax3.set_xlim(0, max(values) * 1.25)
legend_els = [
    mpatches.Patch(color='#4a90d9', label='Baseline'),
    mpatches.Patch(color='#e74c3c', label='Density drops'),
    mpatches.Patch(color='#2ecc71', label='Density rises'),
]
ax3.legend(handles=legend_els, fontsize=8, loc='lower right')
ax3.spines[['top','right']].set_visible(False)

fig.suptitle(f'What happens when {id_to_name.get(keystone, keystone)} is removed? — {FILM_TITLE}\nProtagonist: {PROTAGONIST_LABEL}; keystone = highest-betweenness non-lead',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(REMOVAL_PLOT_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved → {REMOVAL_PLOT_PATH}')


Saved → ..\data\processed\toy_story_1995\figures\removal_analysis.png


C:\Users\majoa\AppData\Local\Temp\ipykernel_19460\4103454109.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Summary stats (for methodology paragraph)

In [17]:
weights = [G[u][v]["weight"] for u,v in G.edges()]

print("=" * 58)
print(f"SUMMARY — {FILM_ID}")
print("=" * 58)
print(f"Nodes                  : {G.number_of_nodes()}")
print(f"Directed edges (≥{MIN_EDGE_COUNT})   : {G.number_of_edges()}")
print(f"Total utterances       : {sum(weights)}")
print(f"Graph density          : {nx.density(G):.4f}")
print(f"Reciprocity            : {nx.reciprocity(G):.4f}")
print()
print("Top 5 nodes by betweenness:")
for _, r in summary.head(5).iterrows():
    print(f"  {r.character:20s}  betw={r.betweenness:.4f}  in={r.in_degree}  out={r.out_degree}")
print("=" * 58)

SUMMARY — toy_story_1995
Nodes                  : 12
Directed edges (≥3)   : 24
Total utterances       : 390
Graph density          : 0.1818
Reciprocity            : 0.9167

Top 5 nodes by betweenness:
  toy_story_1995_woody  betw=0.9455  in=9  out=8
  toy_story_1995_andy   betw=0.1818  in=2  out=2
  toy_story_1995_mr_potato_head  betw=0.1818  in=2  out=2
  toy_story_1995_sid    betw=0.1818  in=2  out=2
  toy_story_1995_hamm   betw=0.0000  in=1  out=2


## 14. Film-level feature row (for cross-film analysis)

Everything above describes **one** film. To compare films (by lead gender, by year, against
box office / ratings), we need **one row per film**. This section assembles that row and
writes `film_features.csv` in the film's folder. After running all films, concatenate these
rows into one table for regression.

It combines three kinds of feature:

- **Global graph shape** — `density`, `reciprocity`, `avg_path_len`, `leading_eigenvalue`,
  `mean_clustering`. These summarize the whole network in a single number each.
- **Inferential results** — the homophily and protagonist-betweenness **z-scores / p-values**
  from §10–§11. Because these are already standardized against each film's own null, they are
  directly comparable across films (this is what tames the small-N problem at the film level).
- **Keystone & metadata** — who the structural keystone is, its gender vs. the lead's, the
  removal-fragmentation counts, plus `year` and `lead_gender`.


In [18]:
# ── 14a. Global graph-shape features ────────────────────────────────────────
density         = nx.density(G)
reciprocity     = nx.reciprocity(G)                       # share of ties that go both ways
mean_clustering = nx.average_clustering(G.to_undirected())

# Leading eigenvalue (spectral radius) of the weighted adjacency matrix
A           = nx.to_numpy_array(G, weight="weight")
leading_eig = float(np.max(np.abs(np.linalg.eigvals(A))))

# Average path length on the largest connected component (undirected, unweighted):
# the directed graph is rarely strongly connected, so we use the giant component.
UG    = G.to_undirected()
giant = UG.subgraph(max(nx.connected_components(UG), key=len))
avg_path_len = nx.average_shortest_path_length(giant) if giant.number_of_nodes() > 1 else 0.0

# ── 14b. Pull the inferential results saved by §10–§12 ──────────────────────
hom  = pd.read_csv(HOMOPHILY_NULL_PATH).iloc[0]
bet  = pd.read_csv(BETWEENNESS_NULL_PATH)
pmet = pd.read_csv(PROTAGONIST_METRICS_PATH).iloc[0]
prot_bet = bet[bet["character"] == PROTAGONIST].iloc[0]

# ── 14c. Assemble one row ───────────────────────────────────────────────────
lead_gender     = GENDER.get(PROTAGONIST)
keystone_rung3        = pmet.get("keystone_rung3")
keystone_rung3_gender = pmet.get("keystone_rung3_gender")

row = {
    # metadata
    "film_id":              FILM_ID,
    "year":                 YEAR,
    "protagonist":          PROTAGONIST_LABEL,
    "lead_gender":          lead_gender,
    # global graph shape
    "n_nodes":              G.number_of_nodes(),
    "n_edges":              G.number_of_edges(),
    "density":              round(density, 4),
    "reciprocity":          round(reciprocity, 4),
    "avg_path_len":         round(avg_path_len, 4),
    "leading_eigenvalue":   round(leading_eig, 4),
    "mean_clustering":      round(mean_clustering, 4),
    # homophily inference (§10)
    "homophily_obs":        round(hom["homophily_obs_edge"], 4),
    "homophily_z":          round(hom["z_score"], 3),
    "homophily_p":          round(hom["p_more_same_gender"], 4),
    "protag_samesex":       round(hom["protag_samesex_obs"], 4),
    "protag_samesex_z":     round(hom["protag_samesex_z"], 3),
    "protag_samesex_p":     round(hom["protag_samesex_p_more"], 4),
    # protagonist betweenness inference (§11)
    "protag_betweenness":   round(prot_bet["betw_obs"], 4),
    "protag_betw_z":        round(prot_bet["z_score"], 3),
    "protag_betw_p":        round(prot_bet["p_high"], 4),
    # keystones — three-rung ladder (rung 3 = PRIMARY)
    "keystone_rung1":            pmet.get("keystone_rung1"),
    "keystone_rung1_gender":     pmet.get("keystone_rung1_gender"),
    "keystone_rung2":            pmet.get("keystone_rung2"),
    "keystone_rung2_gender":     pmet.get("keystone_rung2_gender"),
    "keystone_rung3":            keystone_rung3,
    "keystone_rung3_gender":     keystone_rung3_gender,
    "keystone_changes_at_rung2": int(pmet.get("keystone_changes_at_rung2", 0)) if pd.notna(pmet.get("keystone_changes_at_rung2")) else None,
    "keystone_changes_at_rung3": int(pmet.get("keystone_changes_at_rung3", 0)) if pd.notna(pmet.get("keystone_changes_at_rung3")) else None,
    # primary-keystone aliases (point at rung 3) for backward compatibility
    "keystone":                  keystone_rung3,
    "keystone_gender":           keystone_rung3_gender,
    "keystone_diff_gender":      int(keystone_rung3_gender != lead_gender) if pd.notna(keystone_rung3_gender) else None,
    # removal stats
    "components_after_removal":  int(pmet["components_after_removal"]),
    "n_isolated_after_removal":  int(pmet["n_isolated_after_removal"]),
}

film_features = pd.DataFrame([row])
film_features.to_csv(FILM_FEATURES_PATH, index=False)
print(f"Saved → {FILM_FEATURES_PATH}")
print(film_features.T.to_string(header=False))

# ── 14d. Upsert this film's row into the master cross-film table ─────────────
# Running 06 for any film refreshes ONLY that film's row(s) in film_features_all.csv,
# keyed by (film_id, protagonist) so co-leads (e.g. monsters_inc Mike & Sulley) coexist.
ALL_PATH = FEAT_ROOT / "film_features_all.csv"
if ALL_PATH.exists():
    master = pd.read_csv(ALL_PATH)
    master = master[~((master["film_id"] == FILM_ID) &
                      (master["protagonist"] == PROTAGONIST_LABEL))]
    master = pd.concat([master, film_features], ignore_index=True)
else:
    master = film_features.copy()
master = master.sort_values(["lead_gender", "year", "film_id", "protagonist"]).reset_index(drop=True)
master.to_csv(ALL_PATH, index=False)
print(f"Updated master table -> {ALL_PATH}  ({len(master)} rows total)")

Saved → ..\data\processed\toy_story_1995\film_features.csv
film_id                    toy_story_1995
year                                 1995
protagonist                         Woody
lead_gender                             M
n_nodes                                12
n_edges                                24
density                            0.1818
reciprocity                        0.9167
avg_path_len                       2.0758
leading_eigenvalue               103.6528
mean_clustering                     0.338
homophily_obs                        0.75
homophily_z                        -0.252
homophily_p                           1.0
protag_samesex                     0.8889
protag_samesex_z                    4.225
protag_samesex_p                   0.0055
protag_betweenness                 0.9455
protag_betw_z                       1.412
protag_betw_p                      0.0455
keystone_rung1                       Andy
keystone_rung1_gender                   M
keystone_rung2   